![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)

# <h1><center>Electrical Motor Visualizer on Colab <a href="https://colab.research.google.com/github/robomotic/mujoco/blob/motors/python/examples/electrical/demo_visualizer_colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a></center></h1>

This notebook will:

1. fetch the remote `motors` branch,
2. build MuJoCo from source,
3. run the electrical visualizer demo, and
4. expose a browser port so the viewer can be opened inside Google Colab.

> If you are using Colab, run the cells top-to-bottom. The viewer will appear through an embedded noVNC page.

from pathlib import Path

REPO_URL = 'https://github.com/robomotic/mujoco.git'
BRANCH = 'motors'
REPO_DIR = Path('/content/mujoco')
BUILD_DIR = REPO_DIR / 'build_colab'
VNC_PORT = 6080
DISPLAY_ID = ':1'

print(f'Repo:   {REPO_URL}')
print(f'Branch: {BRANCH}')
print(f'Build:  {BUILD_DIR}')
print(f'Port:   {VNC_PORT}')

In [ ]:
%%bash
set -euxo pipefail
apt-get update
DEBIAN_FRONTEND=noninteractive apt-get install -y \
  git cmake ninja-build build-essential pkg-config patchelf \
  libgl1-mesa-dev libegl1-mesa-dev libgles2-mesa-dev \
  libglfw3-dev libxrandr-dev libxinerama-dev libxcursor-dev \
  libxi-dev libxxf86vm-dev libosmesa6-dev mesa-utils \
  xvfb fluxbox x11vnc websockify novnc
python3 -m pip install --upgrade pip setuptools wheel
python3 -m pip install --upgrade numpy absl-py etils[epath] glfw pyopengl

In [ ]:
%%bash
set -euxo pipefail
if [ ! -d /content/mujoco/.git ]; then
  git clone https://github.com/robomotic/mujoco.git /content/mujoco
fi
cd /content/mujoco
git remote set-url origin https://github.com/robomotic/mujoco.git
git fetch --all --prune
git checkout motors
git pull --ff-only origin motors
git status --short --branch

In [ ]:
%%bash
set -euxo pipefail
cd /content/mujoco
cmake -S . -B build_colab -G Ninja -DCMAKE_BUILD_TYPE=Release
cmake --build build_colab -j"$(nproc)"
export MUJOCO_PATH="$PWD"
export MUJOCO_PLUGIN_PATH="$PWD/build_colab/plugin"
python3 -m pip install -e python

In [ ]:
import os
import subprocess
import time
from google.colab import output

os.environ['DISPLAY'] = DISPLAY_ID
os.environ['LIBGL_ALWAYS_SOFTWARE'] = '1'

_bg_processes = globals().get('_bg_processes', {})

def start_once(name, cmd, env=None):
    proc = _bg_processes.get(name)
    if proc is not None and proc.poll() is None:
        print(f'{name} already running (pid={proc.pid})')
        return proc
    proc = subprocess.Popen(cmd, env=env or os.environ.copy(), stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    _bg_processes[name] = proc
    print(f'started {name} (pid={proc.pid})')
    return proc

start_once('xvfb', ['Xvfb', DISPLAY_ID, '-screen', '0', '1440x900x24', '-ac', '+extension', 'GLX', '+render'])
time.sleep(2)
start_once('fluxbox', ['fluxbox'], env={**os.environ, 'DISPLAY': DISPLAY_ID})
start_once('x11vnc', ['x11vnc', '-display', DISPLAY_ID, '-forever', '-shared', '-nopw', '-rfbport', '5901'])
start_once('novnc', ['websockify', '--web=/usr/share/novnc/', str(VNC_PORT), 'localhost:5901'])

print(f'Opening noVNC on port {VNC_PORT}...')
output.serve_kernel_port_as_iframe(VNC_PORT, path='/vnc.html?autoconnect=true&resize=scale', height=720)

In [ ]:
import os
import subprocess

env = os.environ.copy()
env['DISPLAY'] = DISPLAY_ID
env['PYTHONPATH'] = f'{REPO_DIR}/python'
env['LD_LIBRARY_PATH'] = f'{BUILD_DIR}/lib:' + env.get('LD_LIBRARY_PATH', '')
env['MUJOCO_PLUGIN_PATH'] = f'{BUILD_DIR}/plugin'

demo_cmd = [
    'python3',
    'python/examples/electrical/demo_visualizer.py',
    '--light',
    '--steps',
    '5000',
]

demo_proc = subprocess.Popen(demo_cmd, cwd=str(REPO_DIR), env=env)
print(f'Visualizer started with PID {demo_proc.pid}.')
print('Open the embedded noVNC pane above, then press F4 inside the viewer for the sensor panel.')

## Optional helpers

- Re-run the **noVNC** cell if the browser frame disconnects.
- Change `--light` to `--heavy` in the launch cell to run the heavier payload scenario.
- If you want to stop the current viewer, run:

```python
import os, signal
os.kill(demo_proc.pid, signal.SIGTERM)
```